In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression

from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)

import warnings
warnings.filterwarnings("ignore")

In [ ]:
df = pd.read_csv("cleaned_data.csv")
display(df.head())
print(df.shape)
print(df.dtypes)

In [ ]:
y_reg = df["price"]
y_clf = (y_reg > y_reg.median()).astype(int)
X = df.drop(columns=["price"])
print("Regression Target:", y_reg.name)
print("\nClassification Target Distribution:")
print(y_clf.value_counts())

In [ ]:
X = pd.get_dummies(X, drop_first=True)
print("Encoded Feature Matrix Shape:", X.shape)
display(X.head())

In [ ]:
X_train, X_test, y_reg_train, y_reg_test = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)
_, _, y_clf_train, y_clf_test = train_test_split(
    X, y_clf, test_size=0.2, random_state=42
)
print("Training Set:", X_train.shape)
print("Testing Set :", X_test.shape)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Feature scaling completed successfully.")

In [ ]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_reg_train)
y_pred_reg = lr.predict(X_test_scaled)

In [ ]:
mse = mean_squared_error(y_reg_test, y_pred_reg)
r2 = r2_score(y_reg_test, y_pred_reg)

print(f"MSE : {mse:.2f}")
print(f"R²  : {r2:.4f}")

In [ ]:
coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": lr.coef_
})

coef_df["Absolute"] = coef_df["Coefficient"].abs()

coef_df = coef_df.sort_values("Absolute", ascending=False)

display(coef_df)

print("Top 3 Features")
display(coef_df.head(3))

In [ ]:
ridge = Ridge(alpha=1.0)

ridge.fit(X_train_scaled, y_reg_train)

ridge_pred = ridge.predict(X_test_scaled)

ridge_mse = mean_squared_error(y_reg_test, ridge_pred)
ridge_r2 = r2_score(y_reg_test, ridge_pred)

comparison = pd.DataFrame({
    "Model": ["Linear Regression", "Ridge Regression"],
    "MSE": [mse, ridge_mse],
    "R²": [r2, ridge_r2]
})

display(comparison)

In [ ]:
print(y_clf_train.value_counts())

print("\nPercentage")

print((y_clf_train.value_counts(normalize=True) * 100).round(2))

In [ ]:
log_reg = LogisticRegression(max_iter=1000, class_weight="balanced")

log_reg.fit(X_train_scaled, y_clf_train)

y_pred = log_reg.predict(X_test_scaled)
y_prob = log_reg.predict_proba(X_test_scaled)[:, 1]

In [ ]:
cm = confusion_matrix(y_clf_test, y_pred)

print("Confusion Matrix")
print(cm)

print("\nClassification Report")
print(classification_report(y_clf_test, y_pred))

In [ ]:
fpr, tpr, thresholds = roc_curve(y_clf_test, y_prob)

auc = roc_auc_score(y_clf_test, y_prob)

plt.figure(figsize=(7,5))

plt.plot(fpr, tpr, label=f"AUC = {auc:.3f}")
plt.plot([0,1],[0,1],'--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")

plt.legend()

plt.show()

print("AUC:", auc)

In [ ]:
thresholds = [0.30, 0.40, 0.50, 0.60, 0.70]

results = []

for t in thresholds:

    pred = (y_prob >= t).astype(int)

    results.append([
        t,
        precision_score(y_clf_test, pred),
        recall_score(y_clf_test, pred),
        f1_score(y_clf_test, pred)
    ])

threshold_table = pd.DataFrame(
    results,
    columns=["Threshold","Precision","Recall","F1"]
)

display(threshold_table)

In [ ]:
log_reg2 = LogisticRegression(
    C=0.01,
    max_iter=1000,
    class_weight="balanced"
)

log_reg2.fit(X_train_scaled, y_clf_train)

prob2 = log_reg2.predict_proba(X_test_scaled)[:,1]

pred2 = log_reg2.predict(X_test_scaled)

comparison = pd.DataFrame({
    "Model":["C=1.0","C=0.01"],
    "Precision":[
        precision_score(y_clf_test,y_pred),
        precision_score(y_clf_test,pred2)
    ],
    "Recall":[
        recall_score(y_clf_test,y_pred),
        recall_score(y_clf_test,pred2)
    ],
    "AUC":[
        roc_auc_score(y_clf_test,y_prob),
        roc_auc_score(y_clf_test,prob2)
    ]
})

display(comparison)

In [ ]:
np.random.seed(42)

auc_diff = []

for _ in range(500):

    idx = np.random.choice(
        len(y_clf_test),
        size=len(y_clf_test),
        replace=True
    )

    y_true = y_clf_test.iloc[idx]

    auc1 = roc_auc_score(y_true, y_prob[idx])
    auc2 = roc_auc_score(y_true, prob2[idx])

    auc_diff.append(auc1 - auc2)

print("Mean Difference:", np.mean(auc_diff))

print("95% CI:")

print(np.percentile(auc_diff,[2.5,97.5]))

In [ ]:
plt.savefig("roc_curve.png", dpi=300, bbox_inches="tight")